In [2]:
import pandas as pd
import spacy

In [3]:
nlp = spacy.load('en_core_web_sm')

In [11]:
def lower_replace (series: pd.Series):
    output = series.str.lower()  
    output = output.str.replace(r'\[.*?\]', '', regex=True) # remove [...]
    output = output.str.replace(r'[^\w\s]', '', regex=True) # remove punctuation
    return output

def token_lemma_nonstop (text: str):
    doc = nlp(text)
    output = [token.lemma_ for token in doc if not token.is_stop]
    output = ' '.join(output)
    return output

def filter_pos (text, pos_list=['NOUN', 'PROPN']):
    doc = nlp(text)
    output = [token.text for token in doc if token.pos_ in pos_list]
    output = ' '.join(output)
    return output

In [12]:
def nlp_pipeline(series: pd.Series):
    output = lower_replace(series)
    output = output.apply(token_lemma_nonstop)
    output = output.apply(filter_pos)
    return output

In [13]:
# create a list of sentences
data = [
    "When life gives you lemons, make lemonade! 🙂",
    "She bought 2 lemons for $1 at Maven Market.",
    "A dozen lemons will make a gallon of lemonade. [AllRecipes]",
    "lemon, lemon, lemons, lemon, lemon, lemons",
    "He's running to the market to get a lemon — there's a great sale today.",
    "Does Maven Market carry Eureka lemons or Meyer lemons?",
    "An Arnold Palmer is half lemonade, half iced tea. [Wikipedia]",
    "iced tea is my favorite"
]

In [14]:
pd.set_option('display.max_colwidth', None)

In [15]:
data_df = pd.DataFrame(data=data, columns=["sentence"])

In [18]:
text_clean = nlp_pipeline(data_df['sentence'])

In [19]:
text_clean

0                    life lemon lemonade
1                     lemon maven market
2            dozen lemon gallon lemonade
3    lemon lemon lemon lemon lemon lemon
4              s market lemon sale today
5        maven market eureka lemon lemon
6           palmer lemonade half ice tea
7                       ice tea favorite
Name: sentence, dtype: str

In [20]:
pd.to_pickle(text_clean, 'text_clean.pkl')